# 颜色定位 Color_position

In [ ]:
import cv2 as cv
import threading
import random
import ipywidgets as widgets
from IPython.display import display
from color_position import Color_Position
from dofbot_utils.robot_controller import Robot_Controller
from dofbot_utils.fps import FPS
from dofbot_utils.dofbot_config import *

In [ ]:
robot = Robot_Controller()
robot.move_init_pose()
fps = FPS()

In [ ]:
position = Color_Position()
model = 'General'
color_hsv = {"red": ((0, 25, 90), (10, 255, 255)),
             "green": ((53, 36, 40), (80, 255, 255)),
             "blue": ((110, 80, 90), (120, 255, 255)),
             "yellow": ((25, 20, 55), (50, 255, 255))}
color = [[random.randint(0, 255) for _ in range(3)] for _ in range(255)]
HSV_path="/home/jetson/dofbot_ws/src/dofbot_color_follow/scripts/HSV_config.txt"
try: read_HSV(HSV_path, color_hsv)
except Exception as e: 
    print("Read HSV_config Error !!!")
    print(e)

In [ ]:
button_layout = widgets.Layout(width='200px', height='100px', align_self='center')
# 输出控件 Output widget
output = widgets.Output()
# 颜色定位 Color position
color_position = widgets.Button(description='color_position', button_style='success', layout=button_layout)
# 选择颜色 Select color
choose_color = widgets.ToggleButtons(options=['red', 'green', 'blue', 'yellow'], button_style='success',
             tooltips=['Description of slow', 'Description of regular', 'Description of fast'])
# 取消追踪 Cancel tracking
position_cancel = widgets.Button(description='position_cancel', button_style='danger', layout=button_layout)

# 退出 exit
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
# 图像控件 Image widget
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='auto'))
# 垂直布局 Vertical layout
img_box = widgets.VBox([imgbox, choose_color], layout=widgets.Layout(align_self='auto'))
# 垂直布局 Vertical layout
Slider_box = widgets.VBox([color_position,position_cancel,exit_button],
                          layout=widgets.Layout(align_self='auto'))
# 水平布局 Horizontal layout
controls_box = widgets.HBox([img_box, Slider_box], layout=widgets.Layout(align_self='auto'))
# ['auto', 'flex-start', 'flex-end', 'center', 'baseline', 'stretch', 'inherit', 'initial', 'unset']

In [ ]:
def color_position_Callback(value):
    global model
    model = 'color_position'

def position_cancel_Callback(value):
    global model
    model = 'General'
def exit_button_Callback(value):
    global model
    model = 'Exit'
color_position.on_click(color_position_Callback)

position_cancel.on_click(position_cancel_Callback)
exit_button.on_click(exit_button_Callback)

In [ ]:
def camera():
    global model
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0)
    capture.set(3, 640)
    capture.set(4, 480)
    capture.set(5, 30)  
    # Be executed in loop when the camera is opened normally 
    # 当摄像头正常打开的情况下循环执行
    while capture.isOpened():
        try:
            _, img = capture.read()
            fps.update_fps()
            if model == 'color_position':
                img, pos = position.process(img, color_hsv[choose_color.value])
                cv.putText(img, choose_color.value, (int(img.shape[0] / 2), 50), cv.FONT_HERSHEY_SIMPLEX, 2, color[random.randint(0, 254)], 2)
                if pos is not None:
                    print("x={}, y={}".format(pos[0], pos[1]))
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            fps.show_fps(img)
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except:capture.release()

In [ ]:
display(controls_box,output)
threading.Thread(target=camera, ).start()